> 梯度不可以通过采样反向传播。

$$
\nabla_\theta \mathbb{E}_{x \sim p_\theta(x)}[f(x)]
$$

- 在数学本质上是为了解决同一个目标：求期望的梯度
    - 参数 $\theta$ 隐藏在概率分布 $p_{\theta}$ 中，而采样操作 $x \sim p_{\theta}(x)$ 本身是不可微的。即无法直接让梯度穿过一个随机采样节点。
- score function estimator vs. pathwise gradient estimator
    - score：$\nabla_{\theta} \log p_{\theta}(x)$（对数似然函数（Log-likelihood）对参数的梯度）
        - 对谁求导就是优化/求解的是谁，这里是模型（参数）
    - Pathwise Gradient: “沿着（可微）路径传导的梯度”
- 下游函数 $f(x)$ 是“黑盒”还是“白盒”？
    - 在强化学习（RL）的语境下，$f(x) \Rightarrow R(\tau)$ （环境给出的累积奖励函数，完全不可导的黑盒）。
        - $\mathbb{E}_{x \sim p_{\theta}} [f(x) \nabla_{\theta} \log p_{\theta}(x)]$
        - $\nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]= \nabla_\theta \int \pi_\theta(\tau) R(\tau) d\tau= \int \pi_\theta(\tau) \nabla_\theta \log \pi_\theta(\tau) R(\tau) d\tau= \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau) \nabla_\theta \log \pi_\theta(\tau)]$。
        - 目标梯度的本质是将“轨迹的黑盒奖励 $R(\tau)$”作为权重，去缩放“生成该轨迹的对数概率梯度 $\nabla_\theta \log \pi_\theta(\tau)$”。
    - VAE 语境下，下游函数 $f(x)$ 是生成模型的“解码器（Decoder）”。既然解码器是我们自己搭建的神经网络，自然是一个完全可导的“白盒”。（编码器（Encoder）输出一个概率分布的参数（通常是多元高斯分布的均值 $\mu$ 和标准差 $\sigma$），随后在这个概率分布中进行随机抽取，得到具体的隐变量 $z$，再将其送入解码器（Decoder）进行重构或生成。）
        - $\nabla_{\theta} \mathbb{E}_{x \sim p_{\theta}(x)} [f(x)] = \nabla_{\theta} \mathbb{E}_{\epsilon \sim p(\epsilon)} [f(g(\theta, \epsilon))]=\mathbb{E}_{\epsilon \sim p(\epsilon)} [\nabla_{\theta} f(g(\theta, \epsilon))]=\mathbb{E}_{\epsilon \sim p(\epsilon)} \left[ \nabla_x f(x) \cdot \nabla_{\theta} g(\theta, \epsilon) \right]$
            - $\frac{d}{dx}[f(g(x))] = f'(g(x)) \cdot g'(x)$
        - $p_\theta(x) \Rightarrow q_\phi(z|x_{data})$ （近似后验分布，通常假设为对角高斯分布。注意：这里的 $x_{data}$ 代表输入的真实图像，属于外部条件常量，与基石公式中的采样变量 $x$ 无关）。
        - $f(x) \Rightarrow L(z)$ （解码器的目标函数，通常是重建似然 $\log p_\theta(x_{data}|z)$，由于解码器是神经网络，这是一个完全可导的白盒函数）。
        - $\nabla_\phi \mathbb{E}_{z \sim q_\phi(z)}[L(z)]$


### Score function/likelihood-ratio estimator (Score Function Trick)

> $a \sim \pi_\theta(a|s)$

$$
\nabla_\theta \mathbb{E}_{a \sim \pi_\theta}[R(a)] = \mathbb{E}_{a \sim \pi_\theta}[R(a) \nabla_\theta \log \pi_\theta(a)]
$$

- Score Function Trick：Policy Gradient，它将环境视为黑盒，完全通过“试错”来调整输出概率（$R(a)$ 不可微）。
    - log derivative trick 变换之后，可求导，可计算，可采样计算 objectives（objective 可反向传播）
    - $\nabla_\theta \log p_\theta(\tau)=\frac{\nabla_\theta p_\theta(\tau)}{p_\theta(\tau)}$，概率的相对变化率，也叫 score。
        - 在统计学中就是标准的“得分函数（Score Function）”。它的数学期望恒为 0，
    - 优势：极度通用。无论是大语言模型生成离散的 Token，还是机器人输出连续的扭矩，只要能采样、能算概率，就能用。
    - 劣势：方差极大。因为它相当于蒙上眼睛在广袤的空间里瞎子摸象，极容易受到极值样本的干扰。
- 类比：就像是一个射击运动员（参数 $\theta$）打出了子弹（采样）。子弹飞在空中受风速等随机影响（黑盒环境与采样随机性），轨迹无法求导计算。教练（优化器）不要求计算子弹轨迹的导数，而是只看最终上靶的环数（$R$），然后直接指导运动员：“下次你的枪口（概率分布的中心）往左偏一点”。

```python
dist = Categorical(probs)
a = dist.sample()
log_prob = dist.log_prob(a)

loss = -log_prob * G
loss.backward()
```

### Pathwise derivative estimator (Reparameter trick)

> $z \sim \mathcal{N}(\mu_\theta, \sigma_\theta^2)$

In [10]:
import torch

# 1. 初始化分布参数，并显式要求计算图追踪其梯度
mu = torch.tensor([0.0], requires_grad=True)
sigma = torch.tensor([1.0], requires_grad=True)

# 2. 构建正态分布对象
dist = torch.distributions.Normal(mu, sigma)

# 3. 标准采样 (不可微)
x_sample = dist.sample()

x_sample.requires_grad

False

In [11]:
loss_sample = x_sample.sum() * 2.0
loss_sample.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [9]:
dist.rsample??

Signature:
dist.rsample(
    sample_shape: Union[torch.Size, list[int], tuple[int, ...]] = torch.Size([]),
) -> torch.Tensor
Docstring:
Generates a sample_shape shaped reparameterized sample or sample_shape
shaped batch of reparameterized samples if the distribution parameters
are batched.
Source:   
    def rsample(self, sample_shape: _size = torch.Size()) -> Tensor:
        shape = self._extended_shape(sample_shape)
        eps = _standard_normal(shape, dtype=self.loc.dtype, device=self.loc.device)
        return self.loc + eps * self.scale
File:      ~/miniconda3/envs/verl/lib/python3.12/site-packages/torch/distributions/normal.py
Type:      method

- 传统采样 $x \sim \mathcal{N}(\mu, \sigma^2)$ 是一个不可导的黑盒。
- $x = g(\mu, \sigma, \epsilon) = \mu + \sigma \cdot \epsilon \quad \text{其中} \quad \epsilon \sim \mathcal{N}(0, 1)$

In [18]:
# 4. 重参数化采样 (可微)
x_rsample = dist.rsample()
x_rsample.retain_grad()
x_rsample, x_rsample.requires_grad

(tensor([-1.7795], grad_fn=<AddBackward0>), True)

In [19]:
loss_rsample = x_rsample.sum() * 2.0
loss_rsample.backward()

In [20]:
mu.grad, sigma.grad, x_rsample.grad

(tensor([4.]), tensor([-3.6527]), tensor([2.]))

- 已知 $\text{loss} = 2 \cdot x = 2 \cdot (\mu + \epsilon \cdot \sigma)$。
- 对 $\mu$ 求偏导：$\frac{\partial \text{loss}}{\partial \mu} = 2$
- 对 $\sigma$ 求偏导：$\frac{\partial \text{loss}}{\partial \sigma} = 2 \cdot \epsilon$。这里的梯度值就是底层生成的底噪 $\epsilon$ 放大两倍后的结果。

在变分自编码器（VAE）中，需要在给定的后验分布 $q_\phi(z|x)$ 下优化目标函数 $f(z)$，通常假设 $q_\phi$ 为高斯分布 $\mathcal{N}(\mu, \sigma^2)$。目标依然是求期望的梯度：
$$
\nabla_\phi \mathbb{E}_{z \sim q_\phi(z|x)}[f(z)]
$$
- $f(z) = \log\frac{p_\theta(x,z)}{q_\phi(z|x)}=\log p_\theta(x|z) + \log p(z) - \log q_\phi(z|x)$
- 重参数化技巧引入了一个不依赖于 $\phi$ 的独立底部分布 $p(\epsilon) = \mathcal{N}(0, 1)$，并通过一个确定性且可微的变换函数 $g_\phi(\epsilon, x) = \mu + \sigma \odot \epsilon$ 来生成 $z$。此时，期望的表达形式发生了根本性转移：
$$
\nabla_\phi \mathbb{E}_{z \sim q_\phi}[f(z)] = \nabla_\phi \mathbb{E}_{\epsilon \sim p(\epsilon)}[f(g_\phi(\epsilon, x))]= \mathbb{E}_{\epsilon \sim p(\epsilon)}[\nabla_\phi f(g_\phi(\epsilon, x))]
$$

- reconstruction term：$\mathbb{E}_{z \sim q_\phi(z \mid x)}
\left[
\log p_\theta(x \mid z)
\right]
=\mathbb{E}_{\epsilon \sim \mathcal{N}(0, I)}
\left[\log p_\theta
\left(x\mid\mu_\phi(x)+\sigma_\phi(x) \odot \epsilon
\right)
\right]$
- 对 $\phi$ 求导：$\nabla_\phi\mathbb{E}_{\epsilon}
\left[\log p_\theta(x \mid g_\phi(x, \epsilon))\right]=\mathbb{E}_{\epsilon}\left[\nabla_\phi\log p_\theta(x \mid g_\phi(x, \epsilon))\right]$
    - $\nabla_\phi\log p_\theta(x \mid z)=\left(
\nabla_z\log p_\theta(x \mid z)\right)\frac{\partial z}{\partial \phi}$, $z=\mu_\phi(x)+\sigma_\phi(x) \odot \epsilon$
        - $\frac{\partial z}{\partial \phi}=\frac{\partial \mu_\phi(x)}{\partial \phi}+\epsilon \odot\frac{\partial \sigma_\phi(x)}{\partial \phi}$
- $\nabla_\phi\mathbb{E}_{q_\phi(z \mid x)}\left[
\log p_\theta(x \mid z)
\right]=\mathbb{E}_{\epsilon}\left[\left(\nabla_z
\log p_\theta(x \mid z)\right)\left(\frac{\partial \mu_\phi(x)}{\partial \phi}+\epsilon \odot
\frac{\partial \sigma_\phi(x)}{\partial \phi}\right)\right]$
    -  $\nabla_z \log p_\theta(x \mid z)$：解码器（Decoder）的梯度回传
    -  $\frac{\partial \mu_\phi(x)}{\partial \phi}$ 与 $\frac{\partial \sigma_\phi(x)}{\partial \phi}$：编码器（Encoder）的梯度回传
- 这就是 VAE 中 encoder 能够从 reconstruction loss 获得梯度的关键。

```python
epsilon = torch.randn_like(std)  # 不参与梯度的叶子节点
z = mu + epsilon * sigma          # 确定性的算数运算
```

#### Pathwise derivative

- https://docs.pytorch.org/docs/2.12/distributions.html#pathwise-derivative

```python
params = policy_network(state)
m = Normal(*params)
# Any distribution with .has_rsample == True could work based on the application
action = m.rsample()
next_state, reward = env.step(action)  # Assuming that reward is differentiable
loss = -reward
loss.backward()
```